# 02 — Sentiment distribution & next-hour return correlation

**Disclaimer:** Decision-support only. Not financial advice.

Goals:
1. Visualize the FinBERT sentiment-score distribution per symbol.
2. Inspect time-of-day patterns (are we scraping a quiet hour?).
3. Compute the Spearman rank correlation between per-5-min aggregate sentiment
   and the next-hour price return.

We expect a small but non-zero correlation in the **right direction on average**
and a large amount of noise. The point is to *quantify* the noise, not to claim
predictive power.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("PROJECT_ROOT", "/Users/vishnoiprem/PycharmProjects/data-etl-ml/ai-engineering/realtime-crypto-sentiment"))
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import text
from src.store import db
print("project:", PROJECT)

In [ ]:
# Pull all scored tweets
tweets = pd.read_sql(
    text("SELECT ts, symbol, sentiment_score, author FROM tweets "
         "WHERE sentiment_score IS NOT NULL"),
    db.get_engine(),
)
tweets["ts"] = pd.to_datetime(tweets["ts"])
print(f"{len(tweets):,} scored tweets")
tweets.head()

In [ ]:
# 1) Distribution per symbol
fig = px.histogram(
    tweets, x="sentiment_score", color="symbol", nbins=40,
    barmode="overlay", opacity=0.6,
    title="FinBERT sentiment score distribution per symbol",
)
fig.update_layout(height=400)
fig.show()

print("Per-symbol summary stats:")
tweets.groupby("symbol")["sentiment_score"].agg(["count", "mean", "std"]).round(3)

In [ ]:
# 2) Time-of-day pattern (UTC hour)
tweets["hour_utc"] = tweets["ts"].dt.hour
hourly = tweets.groupby("hour_utc").size().rename("tweets")
fig = px.bar(x=hourly.index, y=hourly.values, labels={"x": "UTC hour", "y": "tweets"},
             title="Tweet volume by UTC hour")
fig.update_layout(height=300)
fig.show()

# A quiet-hour warning: < 10 tweets in the active window
quiet_hours = hourly[hourly < 10].index.tolist()
if quiet_hours:
    print(f"Quiet hours (<10 tweets/day): {quiet_hours}")

In [ ]:
# 3) Spearman corr: 5-min sentiment aggregate → next-hour return
def next_hour_return(sym: str) -> pd.DataFrame:
    # 5-min mean sentiment
    sents = tweets[tweets["symbol"] == sym].copy()
    sents = sents.set_index("ts").sort_index()
    sent_5m = sents["sentiment_score"].resample("5min").mean().dropna()
    sent_5m.name = "sent_5m"
    # Next-hour close - current close, computed from prices_1m
    if db.get_engine().dialect.name == "postgresql":
        prices = pd.read_sql(
            text("SELECT bucket, close FROM prices_1m WHERE symbol=:s ORDER BY bucket"),
            db.get_engine(), params={"s": sym},
        )
    else:
        prices = pd.read_sql(
            text("SELECT ts AS bucket, price_usd AS close FROM prices WHERE symbol=:s ORDER BY ts"),
            db.get_engine(), params={"s": sym},
        )
    prices["bucket"] = pd.to_datetime(prices["bucket"])
    prices = prices.set_index("bucket").sort_index()
    # For each 5-min sentiment point, find the close at that minute and at +1h
    merged = pd.DataFrame(index=sent_5m.index)
    merged["sent_5m"] = sent_5m
    merged["px_now"] = prices["close"].reindex(merged.index, method="nearest")
    merged["px_next_hour"] = prices["close"].shift(-12).reindex(merged.index, method="nearest")  # 12 * 5min
    merged = merged.dropna()
    if len(merged) < 10:
        return None
    merged["next_hour_return_%"] = (merged["px_next_hour"] / merged["px_now"] - 1) * 100
    return merged

results = []
for sym in sorted(tweets["symbol"].unique()):
    df = next_hour_return(sym)
    if df is None or len(df) < 10:
        print(f"{sym}: insufficient data ({len(df) if df is not None else 0} pts)")
        continue
    rho, p = df["sent_5m"].corr(df["next_hour_return_%"], method="spearman")
    results.append({"symbol": sym, "n": len(df), "spearman": rho, "p_value": p,
                    "mean_return_%": df["next_hour_return_%"].mean(),
                    "std_return_%": df["next_hour_return_%"].std()})

corr_df = pd.DataFrame(results).set_index("symbol").round(4) if results else pd.DataFrame()
corr_df

## How to read this

- A Spearman of `+0.05` to `+0.15` is **typical and meaningful** at minute-5
  granularity: a 1σ positive sentiment burst is associated with a small
  positive next-hour drift, on average.
- A Spearman near `0` means the signal is dominated by noise in the current
  regime — which is often the case outside of macro events.
- A Spearman **negative** would be a red flag: it would mean the signal is
  systematically *wrong* (e.g. contrarian effect, or the data is upside-down).

## Caveats

- Spearman ignores magnitude. A `+0.10` correlation on a noisy series does not
  translate linearly to PnL.
- The next-hour return includes drift, so a positive correlation on its own
  does not imply tradeable alpha after fees.
- Sample size matters. With <100 points per symbol, p-values are unreliable.

## What to do with this

- If correlations are near zero across all symbols, **the signal is not currently
  adding value** — focus on regime detection or different features.
- If one symbol is strongly correlated, study *why* (is it thinly traded, news-
  driven, or just overfit to a single event?). Be skeptical.
- If correlations are all positive and small, the model is working as designed
  for a baseline — iterate from there.
